In [1]:
from tqdm import tqdm
import json
import gzip
import pickle
import pandas as pd
from pyterrier_t5 import MonoT5ReRanker
import pickle
import gzip

/opt/miniconda3/envs/tmp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
import pandas as pd
import pyterrier as pt
import math
import warnings
import itertools
import pyterrier as pt
from collections import defaultdict
from pyterrier.model import add_ranks
import torch
from torch.nn import functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration, MT5ForConditionalGeneration
import os
from tqdm import tqdm
import json
import gzip
import pickle
import argparse
from pyterrier_t5 import MonoT5ReRanker
from datasets import DownloadManager
import ir_datasets
import random



# parser = argparse.ArgumentParser(description="A simple argument parsing example.")

# parser.add_argument('--strategy', type=str, help="Model to use for scoring")
# args = parser.parse_args()

class Args():
    def __init__(self):
        self.strategy = "strategy1"

args = Args()

def read_collection(collection_path: str, text_fields=["text"]):
    doc_dict = {}
    if collection_path.startswith(IRDS_PREFIX):
        irds_name = collection_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for doc in tqdm(
            dataset.docs_iter(),
            desc=f"Loading doc collection from ir_datasets: {irds_name}",
        ):
            doc_id = doc.doc_id
            texts = [getattr(doc, field) for field in text_fields]
            text = " ".join(texts)
            doc_dict[doc_id] = text
    elif collection_path.startswith(HFG_PREFIX):
        hfg_name = collection_path.replace(HFG_PREFIX, "")
        dataset = load_dataset(hfg_name)
        for row in tqdm(
            dataset["passage"],
            desc=f"Loading data from HuggingFace datasets: {hfg_name}",
        ):
            doc_dict[row["id"]] = row["text"]
    else:
        with open(collection_path, "r") as f:
            for line in tqdm(f, desc=f"Reading doc collection from {collection_path}"):
                doc_id, doc_text = line.strip().split("\t")
                doc_dict[doc_id] = doc_text
    return doc_dict


def read_queries(queries_path: str, text_fields=["text"]):
    queries = []
    if queries_path.startswith(IRDS_PREFIX):
        irds_name = queries_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for query in tqdm(
            dataset.queries_iter(),
            desc=f"Loading queries from ir_datasets: {queries_path}",
        ):
            query_id = query.query_id
            texts = [getattr(query, field) for field in text_fields]
            text = " ".join(texts)
            queries.append((query_id, text))
    else:
        with open(queries_path, "r") as f:
            for line in tqdm(f, desc=f"Reading queries from {queries_path}"):
                query_id, query_text = line.strip().split("\t")
                queries.append((query_id, query_text))
    return queries


def read_qrels(qrels_path: str, rel_threshold=0):
    qid2pos = {}
    if qrels_path.startswith(IRDS_PREFIX):
        irds_name = qrels_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for qrel in dataset.qrels_iter():
            if qrel.relevance > rel_threshold:
                qid, did = qrel.query_id, qrel.doc_id
                if not qid in qid2pos:
                    qid2pos[qid] = []
                qid2pos[qid].append(did)
    else:
        qrels = json.load(open(qrels_path, "r"))
        for qid in qrels:
            qid2pos[str(qid)] = [str(did) for did in qrels[qid]]
    return qid2pos


file_map = {
    "sentence-transformers/msmarco-hard-negatives": "https://huggingface.co/datasets/sentence-transformers/msmarco-hard-negatives/resolve/main/cross-encoder-ms-marco-MiniLM-L-6-v2-scores.pkl.gz"
}


def read_ce_score(ce_path: str):
    if ce_path.startswith(HFG_PREFIX):
        hf_name = ce_path.replace(HFG_PREFIX, "")
        _url = file_map[hf_name]
        dl_manager = DownloadManager()
        ce_path = dl_manager.download(_url)
    res = {}
    with gzip.open(ce_path, "rb") as f:
        data = pickle.load(f)
        for qid in tqdm(data, desc=f"Preprocessing data from {ce_path}"):
            res[str(qid)] = {str(did): data[qid][did] for did in data[qid]}
    return res



IRDS_PREFIX = "irds"
HFG_PREFIX = "hf"
queries = read_queries( '/nfs/primary/SPLADE/splade_sans/data/training_queries.tsv')
qrels = read_qrels(f'/nfs/primary/SPLADE/splade_sans/data/{args.strategy}_training_qrels.json')
doc_dict = read_collection('/nfs/primary/SPLADE/splade_sans/data/text.tsv')
ce_score = read_ce_score(f'/nfs/primary/SPLADE/splade_class/get_ce_scores/ce_score_{args.strategy}.pkl.gz')


if args.strategy == "strategy1":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_randomneg_synth_data_monot5_sas_best-1"
elif args.strategy == "strategy2":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_1bm25_proportionate_neg_synth_data_monot5_sas_random_negs_best-31"
elif args.strategy == "strategy3":
    model_path = "/nfs/primary/sas_cross_encoder/trained_models/ohsumed_synth_data_proportionate_negs_sas_easy_negs_best-21"
else:
    raise ValueError(f"Invalid strategy: {args.model_choice}")


monot5_scorer = MonoT5ReRanker(model = model_path, batch_size = 8, verbose = False)

Reading queries from /nfs/primary/SPLADE/splade_sans/data/training_queries.tsv: 334070it [00:00, 1593013.91it/s]
Reading doc collection from /nfs/primary/SPLADE/splade_sans/data/text.tsv: 334134it [00:00, 512742.49it/s]
Preprocessing data from /nfs/primary/SPLADE/splade_class/get_ce_scores/ce_score_strategy1.pkl.gz: 100%|██████████| 334070/334070 [00:00<00:00, 428883.63it/s]


In [36]:
def __getitem__(idx):
    train_group_size = 2
    q_id, q_text = queries[idx]
    pos_index = qrels[q_id][0]
    neg_index = qrels[q_id][1]
    pos_doc = doc_dict[pos_index]
    neg_doc = doc_dict[neg_index]
    pos_score = ce_score[q_id][pos_index]
    neg_score = ce_score[q_id][neg_index]
    return (q_text, [pos_doc, neg_doc], [pos_score, neg_score])

('does acetaldehyde react with ribonuclease A', ['The binding of acetaldehyde to the active site of ribonuclease: alterations in catalytic activity and effects of phosphate. Ribonuclease A was reacted with [1-13C,1,2-14C]acetaldehyde and sodium cyanoborohydride in the presence or absence of 0.2 M phosphate. After several hours of incubation at 4 degrees C (pH 7.4) stable acetaldehyde-RNase adducts were formed, and the extent of their formation was similar regardless of the presence of phosphate. Although the total amount of covalent binding was comparable in the absence or presence of phosphate, this active site ligand prevented the inhibition of enzymatic activity seen in its absence. This protective action of phosphate diminished with progressive ethylation of RNase, indicating that the reversible association of phosphate with the active site lysyl residue was overcome by the irreversible process of reductive ethylation. Modified RNase was analysed using 13C proton decoupled NMR spec